In [0]:

# Configuração do ambiente
catalog = "meu_catalog"
schema = "default"
volume = "inputs"

volume_path = f"/Volumes/{catalog}/{schema}/{volume}"
bronze_schema = f"{catalog}.bronze"

print(f"Volume: {volume_path}")
print(f"Schema Bronze: {bronze_schema}")

In [0]:
# Conferir os arquivos disponíveis no Volume
display(dbutils.fs.ls(volume_path))

In [0]:
# Criar o database/schema da camada Bronze

spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {bronze_schema}
""")

print(f"Schema disponível: {bronze_schema}")

display(
    spark.sql("SHOW SCHEMAS IN meu_catalog")
)

In [0]:
TABELAS_BRONZE_RESET = [
    "tb_movies_info", "tb_movies_financials", "tb_movies_metrics",
    "tb_credits_and_tags", "tb_movies_reviews", "tb_cotacao_dolar",
]

for tabela in TABELAS_BRONZE_RESET:
    spark.sql(f"DROP TABLE IF EXISTS {bronze_schema}.{tabela}")
    print(f"Removida (se existia): {bronze_schema}.{tabela}")

In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
# movies_info_TMDB_IMDB.csv -> tabela tb_movies_info

df_movies_info = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{volume_path}/movies_info_TMDB_IMDB.csv")
)

df_movies_info_bronze = (
    df_movies_info
    .withColumn("ingestion_datetime", current_timestamp())
)

(
    df_movies_info_bronze.write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{bronze_schema}.tb_movies_info")
)

print(f"Tabela gravada: {bronze_schema}.tb_movies_info")

# Validação rápida: tb_movies_info

display(
    spark.sql(f"""
        SELECT *
        FROM {bronze_schema}.tb_movies_info
        LIMIT 10
    """)
)


In [0]:
# movies_financials_IMDB_TMDB.csv -> tabela tb_movies_financials

df_movies_financials = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{volume_path}/movies_financials_IMDB_TMDB.csv")
)

df_movies_financials_bronze = (
    df_movies_financials
    .withColumn("ingestion_datetime", current_timestamp())
)

(
    df_movies_financials_bronze.write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{bronze_schema}.tb_movies_financials")
)

print(f"Tabela gravada: {bronze_schema}.tb_movies_financials")

# Validação rápida: tb_movies_financials

display(
    spark.sql(f"""
        SELECT *
        FROM {bronze_schema}.tb_movies_financials
        LIMIT 10
    """)
)

In [0]:
# movies_metrics_IMDB_TMDB.csv -> tabela tb_movies_metrics
# Este arquivo tem linhas com aspas nao fechadas (texto de sinopse vazado nas
# colunas numericas) que quebram o parsing padrao SEM gerar erro nem aviso --
# confirmado rodando contra o CSV real: sem schema/PERMISSIVE explicitos, o
# Spark grava texto deslocado dentro de colunas numericas silenciosamente, e
# parte das linhas corrompidas vira registro duplicado. O schema + PERMISSIVE
# + _corrupt_record abaixo sinalizam essas linhas em vez de mascarar o problema.

from pyspark.sql.types import StructType, StructField, StringType

colunas_metrics = ["id", "popularity", "vote_average", "vote_count", "averageRating", "numVotes"]
schema_metrics = StructType(
    [StructField(c, StringType(), True) for c in colunas_metrics]
    + [StructField("_corrupt_record", StringType(), True)]
)

df_movies_metrics = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(schema_metrics)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .load(f"{volume_path}/movies_metrics_IMDB_TMDB.csv")
)

df_movies_metrics_bronze = (
    df_movies_metrics
    .withColumn("ingestion_datetime", current_timestamp())
)

(
    df_movies_metrics_bronze.write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{bronze_schema}.tb_movies_metrics")
)

df_validacao_metrics = spark.table(f"{bronze_schema}.tb_movies_metrics")
total = df_validacao_metrics.count()
corrompidos = df_validacao_metrics.filter(df_validacao_metrics._corrupt_record.isNotNull()).count()
print(f"Tabela gravada: {bronze_schema}.tb_movies_metrics ({total} linhas, {corrompidos} sinalizadas em _corrupt_record)")


# Validação rápida: tb_movies_metrics

display(
    spark.sql(f"""
        SELECT *
        FROM {bronze_schema}.tb_movies_metrics
        LIMIT 10
    """)
)

In [0]:
# credits_and_tags_IMDB_TMDB.csv -> tabela tb_credits_and_tags
# Mesmo problema de aspas nao fechadas do arquivo de metrics, aqui vazando texto
# de sinopse/tagline nas colunas de genres/production_companies. Mesmo tratamento.

colunas_credits = ["id", "genres", "production_companies", "production_countries",
                   "spoken_languages", "keywords", "directors", "writers", "cast"]
schema_credits = StructType(
    [StructField(c, StringType(), True) for c in colunas_credits]
    + [StructField("_corrupt_record", StringType(), True)]
)

df_credits_and_tags = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(schema_credits)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .load(f"{volume_path}/credits_and_tags_IMDB_TMDB.csv")
)

df_credits_and_tags_bronze = (
    df_credits_and_tags
    .withColumn("ingestion_datetime", current_timestamp())
)

(
    df_credits_and_tags_bronze.write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{bronze_schema}.tb_credits_and_tags")
)

df_validacao_credits = spark.table(f"{bronze_schema}.tb_credits_and_tags")
total = df_validacao_credits.count()
corrompidos = df_validacao_credits.filter(df_validacao_credits._corrupt_record.isNotNull()).count()
print(f"Tabela gravada: {bronze_schema}.tb_credits_and_tags ({total} linhas, {corrompidos} sinalizadas em _corrupt_record)")

# Validação rápida: tb_credits_and_tags

display(
    spark.sql(f"""
        SELECT *
        FROM {bronze_schema}.tb_credits_and_tags
        LIMIT 10
    """)
)

In [0]:
# movies_reviews.csv -> tabela tb_movies_reviews

df_movies_reviews = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{volume_path}/movies_reviews.csv")
)

df_movies_reviews_bronze = (
    df_movies_reviews
    .withColumn("ingestion_datetime", current_timestamp())
)

(
    df_movies_reviews_bronze.write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{bronze_schema}.tb_movies_reviews")
)

print(f"Tabela gravada: {bronze_schema}.tb_movies_reviews")

# Validação rápida: tb_movies_reviews

display(
    spark.sql(f"""
        SELECT *
        FROM {bronze_schema}.tb_movies_reviews
        LIMIT 10
    """)
)


In [0]:
from datetime import datetime, timedelta
 
# Data final: hoje
data_fim_default = datetime.now()
 
# Data inicial: 7 dias antes
data_inicio_default = data_fim_default - timedelta(days=7)
 
for nome_widget in ["data_inicio", "data_fim"]:
    try:
        dbutils.widgets.remove(nome_widget)
    except Exception:
        pass
 
dbutils.widgets.text(
    "data_inicio",
    data_inicio_default.strftime("%m-%d-%Y"),
    "Data início (MM-DD-AAAA)"
)
 
dbutils.widgets.text(
    "data_fim",
    data_fim_default.strftime("%m-%d-%Y"),
    "Data fim (MM-DD-AAAA)"
)
 
 
 
data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")
 
print(f"Data início: {data_inicio}")
print(f"Data fim: {data_fim}")


In [0]:
# Consultar a API PTAX do Banco Central
import requests
 
url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo"
    f"(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio}'"
    f"&@dataFinalCotacao='{data_fim}'"
    "&$select=dataHoraCotacao,cotacaoCompra"
    "&$format=json"
)
 
print(url)
 
try:
    response = requests.get(url, timeout=30)
    print(f"Status HTTP: {response.status_code}")
    response.raise_for_status()
except requests.exceptions.RequestException as e:
    print(f"Erro ao chamar a API do BCB: {e}")
    raise
 
dados_cotacao = response.json()
 
if "value" not in dados_cotacao:
    raise ValueError("A resposta da API não contém a chave 'value'.")
 
# Guarda contra intervalo sem pregao (ex.: cai todo em fim de semana/feriado
# prolongado) -> sem isso, spark.createDataFrame([]) quebra por nao conseguir
# inferir schema de uma lista vazia.
if not dados_cotacao["value"]:
    raise ValueError(
        f"Nenhuma cotação retornada para o intervalo {data_inicio} a {data_fim}. "
        "Tente um intervalo maior (a API não cobre finais de semana/feriados)."
    )
 
print(f"Registros retornados: {len(dados_cotacao['value'])}")


In [0]:
# Gravar o retorno da API na Bronze

df_cotacao = spark.createDataFrame(dados_cotacao["value"])
display(df_cotacao)
 
df_cotacao_bronze = (
    df_cotacao
    .withColumn("ingestion_datetime", current_timestamp())
)
 
display(df_cotacao_bronze)
 
(
    df_cotacao_bronze
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("meu_catalog.bronze.tb_cotacao_dolar")
)
  
display(
    spark.sql("""
        SELECT *
        FROM meu_catalog.bronze.tb_cotacao_dolar
        ORDER BY dataHoraCotacao
    """)
)
 
spark.sql("""
    DESCRIBE TABLE meu_catalog.bronze.tb_cotacao_dolar
""").show(truncate=False)
 
display(
    spark.sql("""
        SHOW TABLES IN meu_catalog.bronze
    """)
)


In [0]:
# Conferir quantidade de registros e timestamp de ingestão

tabelas_bronze = [
    "tb_movies_info",
    "tb_movies_financials",
    "tb_movies_metrics",
    "tb_credits_and_tags",
    "tb_movies_reviews",
    "tb_cotacao_dolar"
]

for tabela in tabelas_bronze:
    resultado = spark.sql(f"""
        SELECT
            COUNT(*) AS quantidade_registros,
            MIN(ingestion_datetime) AS primeira_ingestao,
            MAX(ingestion_datetime) AS ultima_ingestao
        FROM {bronze_schema}.{tabela}
    """).collect()[0]

    print(
        f"{tabela}: "
        f"{resultado['quantidade_registros']} registros | "
        f"primeira ingestão: {resultado['primeira_ingestao']} | "
        f"última ingestão: {resultado['ultima_ingestao']}"
    )